<a href="https://colab.research.google.com/github/owennoimor/bco7006_practice/blob/main/assessment_3_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Instructions for programmer**

**Condition 1**:  Course Structure: The course structure is submitted below. The unit’s prerequisite for the enrolment should be considered.

- unit_name

- unit_code

- pre-req/ condition

- semest


**Condition 2**: Students Enrolment background
The students are allowed to enrol in two units however the program should check the students’ enrolment background.
Here is a table as a sample of an input.

- st_name

- st_id

- gender

- unit_passed

- semester


**Condition 3**: Enrolment process
Your program should check if the students are eligible for enrolment, otherwise to send an appropriate message and advice to the student
Enrolment as an output:

- enrol_code

- Unit_code

- St_id

- semester

**Condition4**: some units have special consition(s)

**Input**:
- catalogue
- student
- requested_units
- semester

Test conditions
len(requested)>2 -> reject all (provide again) = error massage

For each unit in requested:
- check prereq (R1) -> fail message
- check special rules (R4) -> fail message

build enrolment record

**Output**:
- list of excepted enrolments + feedback

Build order:

**Step1**
- Input: data types
options:
list (no)
dictionary (yes)
dataclass (yes)

Test condition:

- can_enrol()
- method for the class unit.is_eligible_for() + ElectiveUnit subclass

Loop+cap:
process_request() function

Because you must have OOP -> wrap in a class

-> replace "hard coded" inputs with interactive input

input()


Implicit requirements:

- code should run with no errors
- raise/use exception where necessary
- failure messages

#Condition 1: Course Structure

 The course structure is submitted below. The unit’s prerequisite for the enrolment should be considered.

- unit_name

- unit_code

- pre-req/ condition

- semest

In [1]:
#minimum example

catalogue={
    "BCO7006":{"name":"Coding for BA","prereqs":[],"semester":1,"female_only":0},
    "BCO7000":{"name":"Biz analytics","prereqs":["BCO7006"],"semester":1,"female_only":0},
    "BCO6008":{"name":"Predictive Analytics","prereqs":["BCO7000"],"semester":2,"female_only":0},
    "BCO7007":{"name":"Machine Learning","prereqs":["BCO7006","BCO7000"],"semester":2,"female_only":0},
    "WOM1000":{"name":"Women in STEM","prereqs":[],"semester":0,"female_only":1}
}

#student sample data
alice={"st_name":"Alice","st_id":"S001","gender":"Female","unit_passed":["BCO7000","BCO7000"]}

In [2]:
print("Catalogue keys:", list(catalogue.keys()))
print("\nAlice:", alice)
print("\nPrereq for BCO6008:", catalogue["BCO6008"]["prereqs"])

Catalogue keys: ['BCO7006', 'BCO7000', 'BCO6008', 'BCO7007', 'WOM1000']

Alice: {'st_name': 'Alice', 'st_id': 'S001', 'gender': 'Female', 'unit_passed': ['BCO7000', 'BCO7000']}

Prereq for BCO6008: ['BCO7000']


#Condition 2: Students Enrolment background

The students are allowed to enrol in two units however the program should check the students’ enrolment background.
Here is a table as a sample of an input.

- st_name

- st_id

- gender

- unit_passed

- semester

In [7]:
#more advanced:
#using dataclass

from dataclasses import dataclass, field
from typing import List

@dataclass
class Unit:
    code: str
    name: str
    prereqs: List[str]=field(default_factory=list)
    semester: int=1
    female_only: int=0 #if 0= open to all

@dataclass
class Student:
    st_name: str
    st_id: str
    gender: str
    unit_passed: List[str]=field(default_factory=list)

CATALOGUE=[
        Unit("BCO7006","Coding for BA",prereqs=[],semester=1,female_only=0),
        Unit("BCO7000","Biz analytics",prereqs=["BCO7006"],semester=1,female_only=0),
        Unit("BCO6008","Predictive Analytics",prereqs=["BCO7006"],semester=2,female_only=0),
        Unit("BCO7007","Machine Learning",prereqs=["BCO7000", "BCO7006"],semester=2,female_only=0),
        Unit("WOM1000","Women in STEM",prereqs=[],semester=0,female_only=1)
    ]

In [8]:
def can_enrol(student, unit_code,catalogue):
  """Return (ok,message) for single unit"""
  if unit_code not in catalogue:
    return (False,f"Unit {unit_code} not found")

  unit=catalogue[unit_code]

  #check prereq
  missing=[p for p in unit["prereqs"] if p not in student["unit_passed"]]
  if missing:
    return (False,f"Cannot enrol in {unit_code}. Missing prereqs {','.join(missing)}")

  #check conditions
  if unit["female_only"] and student["gender"].lower()!="female":
    return (False,f"Cannot enrol in {unit_code}. Only for female")
  return (True,f"Enrolment OK for {unit_code}{unit['name']}")


In [13]:
#test on cases - ideally edge cases
bob = Student(st_name="Bob", st_id="S002", gender="male", unit_passed=[])

print(can_enrol(alice , "BCO7006", catalogue))
print(can_enrol(alice , "XZXXXX", catalogue))
print(can_enrol(bob , "BCO7006", catalogue))

(True, 'Enrolment OK for BCO7006Coding for BA')
(False, 'Unit XZXXXX not found')
(True, 'Enrolment OK for BCO7006Coding for BA')


#Condition 3: Enrolment process
Your program should check if the students are eligible for enrolment, otherwise to send an appropriate message and advice to the student
Enrolment as an output:

- enrol_code

- Unit_code

- St_id

- semester

In [14]:
def process_request(student,requested_units,semester,catalogue,enrolments,next_code):
  """Process an enrolment reques, transform 'enrolments ' and return the next code"""
  print(f"Processing request for {student['st_name']}")

#cap checking =no more than 2
  if len(requested_units>2):
    print(f"Rejecting request for {student['st_name']}")
    return next_code

    for unit_code in requested_units:
      ok,message=can_enrol(student,unit_code,catalogue)
      if ok:
        record={
          "enrol_code":next_code,
          "unit_code":unit_code,
          "st_id":student["st_id"],
          "semester":semester
        }
        enrolments.append(record)
        next_code+=1
        print(f"OK:{message}->enrol_code:{next_code}")
      if not ok:
        print(f"Fail")
        return next_code